# Data Cleaning

This notebook merges two raw, independently-sourced datasets into a single, clean, analysis-ready table:

- **World Bank — World Development Indicators**: country-level development indicators (GDP, life expectancy, unemployment, inflation, population, FDI, CO2 emissions, poverty) across 25 years.
- **Heritage Foundation — Index of Economic Freedom**: overall economic freedom score plus 12 sub-components (property rights, government integrity, trade freedom, etc.) by country and year.

The goal is not just to combine the two sources, but to "correct" real-world messiness, such as: inconsistent country names/codes, missing years, methodology changes over time, and structural mismatches between sources.

**Parts covered in this notebook:**
1. Imports, loading, and initial inspection
2. Country name/code standardisation
3. Reshaping from wide to long format
4. Merging both datasets
5. Handling missing values and outliers
6. Exporting the final clean dataset to `data/processed/`

---

## Part I — Imports, Loading, and Initial Info

First things first, I need to load both raw datasets and get some basic information about each dataset.

No transformations or cleaning take place in this section, only initial inspection.

In [1]:
import pandas as pd
import sys

sys.path.append("..")

from src import cleaning


In [2]:
wb_df = pd.read_csv("../data/raw/world_bank_indicators.csv")
ef_df = pd.read_csv("../data/raw/economic_freedom_data.csv")

### World Bank - Initial Inspection

In [3]:
wb_df.head()

,Country Name,Country Code,Series Name,Series Code,2001 [YR2001],2002 [YR2002],2003 [YR2003],2004 [YR2004],2005 [YR2005],2006 [YR2006],...,2016 [YR2016],2017 [YR2017],2018 [YR2018],2019 [YR2019],2020 [YR2020],2021 [YR2021],2022 [YR2022],2023 [YR2023],2024 [YR2024],2025 [YR2025]
0,Afghanistan,AFG,GDP (current US$),NY.GDP.MKTP.CD,2813571753.87253,3825701438.99963,4520946818.54581,5224896718.67782,6203256538.70967,6971758282.29351,...,18116572395.0772,18753456497.8159,18053222687.4126,18799444490.1128,19955929052.1496,14259995441.0759,14497243872.1337,17152234636.8715,17778508875.7396,..
1,Afghanistan,AFG,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,-9.4319740700862,28.6000011706788,8.83227780288267,1.41411799339429,11.2297148272859,5.35740324748592,...,2.26031420279821,2.6470032027451,1.18922812944517,3.91160341625552,-2.35110067203466,-20.7388393676343,-6.24017199240269,2.26694373649188,1.87319280045979,..
2,Afghanistan,AFG,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,55.511,56.225,57.171,57.81,58.247,58.553,...,62.646,62.406,62.443,62.941,61.454,60.417,65.617,66.035,66.289,..
3,Afghanistan,AFG,"Unemployment, total (% of total labor force) (...",SL.UEM.TOTL.ZS,7.973,7.867,7.844,7.794,7.878,7.897,...,10.116,11.184,11.192,11.187,11.71,12.006,14.1,14.008,13.687,13.351
4,Afghanistan,AFG,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,..,..,..,..,12.6862687216715,6.78459655001655,...,4.38389195513915,4.97595150553833,0.626149149168847,2.30237251516834,5.60188791482224,5.13320340824963,13.7121023720065,-4.64470870797775,-6.60118564073726,..


In [4]:
wb_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2175 entries, 0 to 2174
Data columns (total 29 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Country Name   2172 non-null   str  
 1   Country Code   2170 non-null   str  
 2   Series Name    2170 non-null   str  
 3   Series Code    2170 non-null   str  
 4   2001 [YR2001]  2170 non-null   str  
 5   2002 [YR2002]  2170 non-null   str  
 6   2003 [YR2003]  2170 non-null   str  
 7   2004 [YR2004]  2170 non-null   str  
 8   2005 [YR2005]  2170 non-null   str  
 9   2006 [YR2006]  2170 non-null   str  
 10  2007 [YR2007]  2170 non-null   str  
 11  2008 [YR2008]  2170 non-null   str  
 12  2009 [YR2009]  2170 non-null   str  
 13  2010 [YR2010]  2170 non-null   str  
 14  2011 [YR2011]  2170 non-null   str  
 15  2012 [YR2012]  2170 non-null   str  
 16  2013 [YR2013]  2170 non-null   str  
 17  2014 [YR2014]  2170 non-null   str  
 18  2015 [YR2015]  2170 non-null   str  
 19  2016 [YR2016]  21

In [5]:
wb_df.describe()

,Country Name,Country Code,Series Name,Series Code,2001 [YR2001],2002 [YR2002],2003 [YR2003],2004 [YR2004],2005 [YR2005],2006 [YR2006],...,2016 [YR2016],2017 [YR2017],2018 [YR2018],2019 [YR2019],2020 [YR2020],2021 [YR2021],2022 [YR2022],2023 [YR2023],2024 [YR2024],2025 [YR2025]
count,2172,2170,2170,2170,2170,2170,2170,2170,2170,2170,...,2170,2170,2170,2170,2170,2170,2170,2170,2170,2170
unique,219,217,10,10,1759,1783,1780,1795,1804,1796,...,1816,1797,1808,1792,1791,1802,1788,1776,1703,947
top,Afghanistan,AFG,GDP (current US$),NY.GDP.MKTP.CD,..,..,..,..,..,..,...,..,..,..,..,..,..,..,..,..,..
freq,10,10,217,217,395,366,360,343,332,335,...,312,329,313,329,339,323,343,366,454,1223


In [6]:
wb_df.describe(include=["object", "str"])

,Country Name,Country Code,Series Name,Series Code,2001 [YR2001],2002 [YR2002],2003 [YR2003],2004 [YR2004],2005 [YR2005],2006 [YR2006],...,2016 [YR2016],2017 [YR2017],2018 [YR2018],2019 [YR2019],2020 [YR2020],2021 [YR2021],2022 [YR2022],2023 [YR2023],2024 [YR2024],2025 [YR2025]
count,2172,2170,2170,2170,2170,2170,2170,2170,2170,2170,...,2170,2170,2170,2170,2170,2170,2170,2170,2170,2170
unique,219,217,10,10,1759,1783,1780,1795,1804,1796,...,1816,1797,1808,1792,1791,1802,1788,1776,1703,947
top,Afghanistan,AFG,GDP (current US$),NY.GDP.MKTP.CD,..,..,..,..,..,..,...,..,..,..,..,..,..,..,..,..,..
freq,10,10,217,217,395,366,360,343,332,335,...,312,329,313,329,339,323,343,366,454,1223


### Economic Freedom Index - Initial Inspection

In [7]:
ef_df.head()

,Country,Index Year,Overall Score,Property Rights,Government Integrity,Judicial Effectiveness,Tax Burden,Government Spending,Fiscal Health,Business Freedom,Labor Freedom,Monetary Freedom,Trade Freedom,Investment Freedom,Financial Freedom
0,Afghanistan,2026,NaN,3.6,13.5,0.0,92.0,90.8,98.4,33.7,44.4,84.9,NaN,NaN,NaN
1,Afghanistan,2025,NaN,7.4,14.1,2.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,2024,NaN,4.9,18.1,4.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,2023,NaN,5.8,5.4,12.7,NaN,NaN,NaN,34.6,45.1,NaN,NaN,NaN,NaN
4,Afghanistan,2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
ef_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5888 entries, 0 to 5887
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Country                 5888 non-null   str    
 1   Index Year              5888 non-null   int64  
 2   Overall Score           5322 non-null   float64
 3   Property Rights         5388 non-null   float64
 4   Government Integrity    5404 non-null   float64
 5   Judicial Effectiveness  1832 non-null   float64
 6   Tax Burden              5347 non-null   float64
 7   Government Spending     5357 non-null   float64
 8   Fiscal Health           1800 non-null   float64
 9   Business Freedom        5386 non-null   float64
 10  Labor Freedom           3891 non-null   float64
 11  Monetary Freedom        5368 non-null   float64
 12  Trade Freedom           5351 non-null   float64
 13  Investment Freedom      5364 non-null   float64
 14  Financial Freedom       5342 non-null   float64
dty

In [9]:
ef_df.describe()

,Index Year,Overall Score,Property Rights,Government Integrity,Judicial Effectiveness,Tax Burden,Government Spending,Fiscal Health,Business Freedom,Labor Freedom,Monetary Freedom,Trade Freedom,Investment Freedom,Financial Freedom
count,5888.000000,5322.000000,5388.000000,5404.000000,1832.000000,5347.000000,5357.000000,1800.000000,5386.000000,3891.000000,5368.000000,5351.000000,5364.000000,5342.000000
mean,2010.537874,59.598572,49.466425,41.702591,46.734061,74.124818,64.663114,63.732389,63.664835,59.441172,72.431967,69.501196,53.842282,49.543242
std,9.227285,11.665205,23.897350,22.586069,23.400168,15.111831,24.472821,31.838235,16.206571,15.197266,15.288104,15.299137,21.343599,19.699747
min,1995.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2003.000000,52.900000,30.000000,26.000000,29.275000,66.500000,51.000000,39.900000,55.000000,50.000000,69.400000,61.600000,40.000000,30.000000
50%,2011.000000,59.600000,50.000000,35.700000,43.600000,76.200000,70.800000,74.700000,64.800000,59.200000,75.600000,72.200000,55.000000,50.000000
75%,2019.000000,67.400000,70.000000,53.300000,62.600000,83.900000,84.100000,90.700000,73.600000,69.000000,80.800000,80.000000,70.000000,67.500000
max,2026.000000,90.500000,100.000000,100.000000,100.000000,100.000000,99.300000,100.000000,100.000000,100.000000,95.400000,95.000000,95.000000,90.000000


In [10]:
ef_df.describe(include=["object", "str"])

,Country
count,5888
unique,186
top,Afghanistan
freq,32


## Part II - Country Name and Code Standardisation

Before merging, both datasets need to represent countries the same way. World Bank and Heritage Foundation are independent sources, so naming conventions often diverge (for example: "Korea, Rep." vs "South Korea", or "Russian Federation" vs "Russia").

This section:
1. Compares the unique country names/codes in each dataset
2. Identifies mismatches (countries present in one source but not matched in the other)
3. Builds a standardised mapping to align both datasets on a common country identifier

In [11]:
wb_df["Country Name"].nunique()

219

In [12]:
ef_df["Country"].nunique()

186

In [13]:
only_in_wb, only_in_ef = cleaning.compare_countries(
    wb_df, ef_df, "Country Name", "Country", "World Bank", "Economic Freedom"
)

Only in World Bank: {'Last Updated: 07/01/2026', 'Venezuela, RB', 'Andorra', 'Myanmar', 'Philippines', 'Cayman Islands', 'Sint Maarten (Dutch part)', 'Congo, Dem. Rep.', 'St. Lucia', 'Bermuda', 'Monaco', 'Antigua and Barbuda', 'Guam', 'Egypt, Arab Rep.', 'French Polynesia', 'St. Martin (French part)', 'Virgin Islands (U.S.)', 'British Virgin Islands', 'St. Vincent and the Grenadines', 'Syrian Arab Republic', 'American Samoa', 'Macao SAR, China', nan, 'Somalia, Fed. Rep.', 'Faroe Islands', 'Gibraltar', 'Northern Mariana Islands', 'Greenland', 'Czechia', 'New Caledonia', 'Viet Nam', 'Gambia, The', 'West Bank and Gaza', 'Korea, Rep.', 'Bahamas, The', 'San Marino', "Korea, Dem. People's Rep.", 'Hong Kong SAR, China', 'Yemen, Rep.', 'Curacao', 'Russian Federation', 'Sao Tome and Principe', 'St. Kitts and Nevis', 'Nauru', 'Isle of Man', "Cote d'Ivoire", 'South Sudan', 'Aruba', 'Puerto Rico (US)', 'Marshall Islands', 'Grenada', 'Lao PDR', 'Slovak Republic', 'Micronesia, Fed. Sts.', 'Iran, Isl

In [14]:
wb_df = cleaning.standardise_country_names(wb_df, "Country Name")

wb_df.head(10)

,Country Name,Country Code,Series Name,Series Code,2001 [YR2001],2002 [YR2002],2003 [YR2003],2004 [YR2004],2005 [YR2005],2006 [YR2006],...,2016 [YR2016],2017 [YR2017],2018 [YR2018],2019 [YR2019],2020 [YR2020],2021 [YR2021],2022 [YR2022],2023 [YR2023],2024 [YR2024],2025 [YR2025]
0,Afghanistan,AFG,GDP (current US$),NY.GDP.MKTP.CD,2813571753.87253,3825701438.99963,4520946818.54581,5224896718.67782,6203256538.70967,6971758282.29351,...,18116572395.0772,18753456497.8159,18053222687.4126,18799444490.1128,19955929052.1496,14259995441.0759,14497243872.1337,17152234636.8715,17778508875.7396,..
1,Afghanistan,AFG,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,-9.4319740700862,28.6000011706788,8.83227780288267,1.41411799339429,11.2297148272859,5.35740324748592,...,2.26031420279821,2.6470032027451,1.18922812944517,3.91160341625552,-2.35110067203466,-20.7388393676343,-6.24017199240269,2.26694373649188,1.87319280045979,..
2,Afghanistan,AFG,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,55.511,56.225,57.171,57.81,58.247,58.553,...,62.646,62.406,62.443,62.941,61.454,60.417,65.617,66.035,66.289,..
3,Afghanistan,AFG,"Unemployment, total (% of total labor force) (...",SL.UEM.TOTL.ZS,7.973,7.867,7.844,7.794,7.878,7.897,...,10.116,11.184,11.192,11.187,11.71,12.006,14.1,14.008,13.687,13.351
4,Afghanistan,AFG,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,..,..,..,..,12.6862687216715,6.78459655001655,...,4.38389195513915,4.97595150553833,0.626149149168847,2.30237251516834,5.60188791482224,5.13320340824963,13.7121023720065,-4.64470870797775,-6.60118564073726,..
5,Afghanistan,AFG,"School enrollment, secondary (% gross)",SE.SEC.ENRR,14.0404100418091,..,13.959529876709,19.2143802642822,20.2735099792481,29.523889541626,...,53.4350395202637,55.5364303588867,57.3578491210938,..,..,..,..,59.6136016845703,..,..
6,Afghanistan,AFG,"Population, total",SP.POP.TOTL,20284307,21378117,22733049,23560654,24404567,25424094,...,34700612,35688935,36743039,37856121,39068979,40000412,40578842,41454761,42647492,43844111
7,Afghanistan,AFG,"Foreign direct investment, net inflows (% of GDP)",BX.KLT.DINV.WD.GD.ZS,0.0241685679088889,1.30694987042884,1.27849325196424,3.577104200584,4.36867310434287,3.41377297323201,...,0.516606084523093,0.274796791572807,0.661572217787196,0.124495985291258,0.0649939571808755,0.144466946606861,..,..,..,..
8,Afghanistan,AFG,Carbon dioxide (CO2) emissions (total) excludi...,EN.GHG.CO2.MT.CE.AR5,0.9402,0.9388,1.0172,0.9024,1.2763,1.4185,...,7.6117,10.3972,11.8099,12.1046,12.1436,12.4864,11.298,11.6486,12.0684,..
9,Afghanistan,AFG,Poverty headcount ratio at $3.00 a day (2021 P...,SI.POV.DDAY,..,..,..,..,..,..,...,..,..,..,..,..,..,..,..,..,..


In [15]:
only_in_wb, only_in_ef = cleaning.compare_countries(
    wb_df, ef_df, "Country Name", "Country", "World Bank", "Economic Freedom"
)

Only in World Bank: {'Last Updated: 07/01/2026', 'Andorra', 'Cayman Islands', 'Sint Maarten (Dutch part)', 'Bermuda', 'Monaco', 'Antigua and Barbuda', 'Guam', 'French Polynesia', 'St. Martin (French part)', 'Virgin Islands (U.S.)', 'British Virgin Islands', 'American Samoa', nan, 'Faroe Islands', 'Gibraltar', 'Northern Mariana Islands', 'Greenland', 'New Caledonia', 'West Bank and Gaza', 'San Marino', 'Curacao', 'Nauru', 'St. Kitts and Nevis', 'Isle of Man', 'South Sudan', 'Aruba', 'Puerto Rico (US)', 'Marshall Islands', 'Grenada', 'Channel Islands', 'Palau', 'Tuvalu', 'Turks and Caicos Islands', 'Data from database: World Development Indicators'}
Only in Economic Freedom: {'Taiwan'}


In [16]:
rows_to_drop = [
    "Data from database: World Development Indicators",
    "Last Updated: 07/01/2026",
    "American Samoa",
    "French Polynesia",
    "Isle of Man",
    "Virgin Islands (U.S.)",
    "Antigua and Barbuda",
    "Andorra",
    "Channel Islands",
    "Greenland",
    "Aruba",
    "Faroe Islands",
    "Monaco",
    "Northern Mariana Islands",
    "Marshall Islands",
    "West Bank and Gaza",
    "Tuvalu",
    "St. Kitts and Nevis",
    "Nauru",
    "Turks and Caicos Islands",
    "San Marino",
    "Bermuda",
    "Grenada",
    "Guam",
    "New Caledonia",
    "British Virgin Islands",
    "Curacao",
    "Palau",
    "St. Martin (French part)",
    "Sint Maarten (Dutch part)",
    "Puerto Rico (US)",
    "Gibraltar",
    "South Sudan",
    "Cayman Islands",
]

In [17]:
wb_df = cleaning.drop_unmatched_countries(wb_df, "Country Name", rows_to_drop)
wb_df = wb_df.dropna(subset=["Country Name"])

wb_df.head(10)

,Country Name,Country Code,Series Name,Series Code,2001 [YR2001],2002 [YR2002],2003 [YR2003],2004 [YR2004],2005 [YR2005],2006 [YR2006],...,2016 [YR2016],2017 [YR2017],2018 [YR2018],2019 [YR2019],2020 [YR2020],2021 [YR2021],2022 [YR2022],2023 [YR2023],2024 [YR2024],2025 [YR2025]
0,Afghanistan,AFG,GDP (current US$),NY.GDP.MKTP.CD,2813571753.87253,3825701438.99963,4520946818.54581,5224896718.67782,6203256538.70967,6971758282.29351,...,18116572395.0772,18753456497.8159,18053222687.4126,18799444490.1128,19955929052.1496,14259995441.0759,14497243872.1337,17152234636.8715,17778508875.7396,..
1,Afghanistan,AFG,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,-9.4319740700862,28.6000011706788,8.83227780288267,1.41411799339429,11.2297148272859,5.35740324748592,...,2.26031420279821,2.6470032027451,1.18922812944517,3.91160341625552,-2.35110067203466,-20.7388393676343,-6.24017199240269,2.26694373649188,1.87319280045979,..
2,Afghanistan,AFG,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,55.511,56.225,57.171,57.81,58.247,58.553,...,62.646,62.406,62.443,62.941,61.454,60.417,65.617,66.035,66.289,..
3,Afghanistan,AFG,"Unemployment, total (% of total labor force) (...",SL.UEM.TOTL.ZS,7.973,7.867,7.844,7.794,7.878,7.897,...,10.116,11.184,11.192,11.187,11.71,12.006,14.1,14.008,13.687,13.351
4,Afghanistan,AFG,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,..,..,..,..,12.6862687216715,6.78459655001655,...,4.38389195513915,4.97595150553833,0.626149149168847,2.30237251516834,5.60188791482224,5.13320340824963,13.7121023720065,-4.64470870797775,-6.60118564073726,..
5,Afghanistan,AFG,"School enrollment, secondary (% gross)",SE.SEC.ENRR,14.0404100418091,..,13.959529876709,19.2143802642822,20.2735099792481,29.523889541626,...,53.4350395202637,55.5364303588867,57.3578491210938,..,..,..,..,59.6136016845703,..,..
6,Afghanistan,AFG,"Population, total",SP.POP.TOTL,20284307,21378117,22733049,23560654,24404567,25424094,...,34700612,35688935,36743039,37856121,39068979,40000412,40578842,41454761,42647492,43844111
7,Afghanistan,AFG,"Foreign direct investment, net inflows (% of GDP)",BX.KLT.DINV.WD.GD.ZS,0.0241685679088889,1.30694987042884,1.27849325196424,3.577104200584,4.36867310434287,3.41377297323201,...,0.516606084523093,0.274796791572807,0.661572217787196,0.124495985291258,0.0649939571808755,0.144466946606861,..,..,..,..
8,Afghanistan,AFG,Carbon dioxide (CO2) emissions (total) excludi...,EN.GHG.CO2.MT.CE.AR5,0.9402,0.9388,1.0172,0.9024,1.2763,1.4185,...,7.6117,10.3972,11.8099,12.1046,12.1436,12.4864,11.298,11.6486,12.0684,..
9,Afghanistan,AFG,Poverty headcount ratio at $3.00 a day (2021 P...,SI.POV.DDAY,..,..,..,..,..,..,...,..,..,..,..,..,..,..,..,..,..


In [18]:
ef_df = cleaning.drop_unmatched_countries(ef_df, "Country", ["Taiwan"])

In [19]:
only_in_wb, only_in_ef = cleaning.compare_countries(
    wb_df, ef_df, "Country Name", "Country", "World Bank", "Economic Freedom"
)

Only in World Bank: set()
Only in Economic Freedom: set()


### Dropping unmatched countries

After standardising country names, a small number of entries remained unmatched between the two datasets:

- **Junk rows:** The World Bank dataset contains metadata footer rows such as "Data from database..." and "Last Updated...", plus a stray `NaN` row. Since these are not countries, they were dropped.
- **Territories and dependencies:** Some countries are present in the World Bank dataset but not on the Economic Freedom Index (e.g. Greenland, Bermuda, Monaco, Puerto Rico). The Index only evaluates sovereign countries with sufficient economic data, so these have no possible match.
- **South Sudan:** A recognised sovereign country, is excluded from the Economic Freedom Index due to insufficient data availability.
- **Taiwan:** Present only in the Economic Freedom Index, has no equivalent entry in the World Bank dataset due to its disputed sovereign status.

These rows were removed from both datasets, since they cannot be meaningfully merged. The final dataset therefore covers only countries present in **both** sources.

---

## Part III -  Reshaping from Wide to Long Format

The World Bank dataset is currently in wide format: each row represents a country-indicator pair, with one column per year (e.g. `2001 [YR2001]`, `2002 [YR2002]`, ...). This structure is convenient for browsing but impractical for merging and analysis.

This section reshapes `wb_df` into long format — one row per country, per year, with each indicator as its own column. This is the structure required to merge with the Economic Freedom Index dataset and to support time-series analysis in the next notebook.

**Steps covered in this section:**
1. Melt the year columns into a single `Year` column
2. Pivot indicators from rows into columns
3. Clean up resulting column names and data types